# Análise Exploratória: City_Tier × Rent

## Objetivo Geral
Investigar como o **nível da cidade** (`City_Tier`) influencia o **valor do aluguel** (`Rent`) pago pelos indivíduos do dataset. A análise busca identificar padrões de distribuição, dispersão e comprometimento de renda com moradia em cada tier urbano.

### Perguntas que este notebook responde:
- Cidades de tier mais alto cobram aluguéis maiores?
- A variação dos aluguéis dentro de um mesmo tier é grande ou pequena?
- Em qual tier as pessoas comprometem mais a renda com moradia?

### Estrutura do notebook:
| Seção | Descrição |
|---|---|
| 0 — Setup | Importação de bibliotecas e carregamento do dataset |
| 1 | Distribuição de registros por City_Tier |
| 2 | Box Plot — Distribuição do Rent por City_Tier |
| 3 | Violin Plot — Densidade da distribuição |
| 4 | Histograma por City_Tier |
| 5 | Estatísticas resumidas (Média, Mediana, Desvio Padrão) |
| 6 | Proporção Rent/Income por City_Tier |
| 7 | Conclusões |


---
## Seção 0 — Setup e Carregamento dos Dados

**Objetivo:** Importar as bibliotecas necessárias, carregar o dataset e preparar a variável `City_Tier` como categórica ordenada (`Tier_1 < Tier_2 < Tier_3`), garantindo que todos os gráficos respeitem essa ordem lógica de hierarquia urbana.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data_3.csv')

# Definir a ordem dos tiers como categoria ordenada.
# Isso garante que os gráficos sempre exibam Tier_1 → Tier_2 → Tier_3,
# refletindo a hierarquia real das cidades (metrópoles > cidades médias > cidades pequenas).
tier_order = ['Tier_1', 'Tier_2', 'Tier_3']
df['City_Tier'] = pd.Categorical(df['City_Tier'], categories=tier_order, ordered=True)
df = df.sort_values('City_Tier')

# Visão geral do dataset e das variáveis de interesse
print(f'Shape: {df.shape}')
print(f"\nDistribuição de City_Tier:\n{df['City_Tier'].value_counts().sort_index()}")
print(f"\nEstatísticas de Rent por City_Tier:")
df.groupby('City_Tier')['Rent'].describe().round(2)

---
## Seção 1 — Distribuição de Registros por City_Tier

**Objetivo:** Verificar quantos registros existem em cada tier antes de qualquer análise comparativa.

**Por que isso importa?**  
Se um tier tiver muito menos registros que os outros, os resultados das seções seguintes para aquele grupo serão menos confiáveis estatisticamente. É essencial saber se as comparações entre tiers partem de grupos balanceados ou não.

In [ ]:
contagem = df['City_Tier'].value_counts().sort_index().reset_index()
contagem.columns = ['City_Tier', 'Count']

fig = px.bar(
    contagem,
    x='City_Tier', y='Count',
    color='City_Tier',
    text='Count',
    title='Quantidade de Registros por City_Tier',
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

# Conta quantos imóveis existem em cada tier e plota um gráfico de barras.
# Tier_2 costuma concentrar a maior parte dos registros (~50%), sendo o mais representativo.
# Essa informação é necessária para interpretar corretamente os resultados das seções seguintes.

---
## Seção 2 — Box Plot: Distribuição do Rent por City_Tier

**Objetivo:** Comparar a distribuição central e a dispersão dos aluguéis entre os três tiers em um único gráfico compacto.

**Por que Box Plot?**  
O box plot resume 5 estatísticas simultaneamente — mínimo, Q1 (25%), mediana, Q3 (75%) e máximo — e destaca os outliers (pontos fora dos bigodes). É ideal para comparar se a centralidade e a variabilidade diferem entre grupos de forma rápida e visual.  
Tier_1 tende a apresentar tanto maior mediana quanto outliers extremos mais elevados.

In [ ]:
# Create a box plot
fig_box = px.box(
    df,
    x='City_Tier',
    y='Rent',
    title='Box Plot of Rent by City Tier',
    labels={'City_Tier': 'City Tier', 'Rent': 'Rent Amount'},
    category_orders={'City_Tier': sorted(df['City_Tier'].unique())}
)
fig_box.show()

---
## Seção 3 — Violin Plot: Densidade da Distribuição

**Objetivo:** Revelar a **forma completa** da distribuição do aluguel em cada tier — algo que o box plot não consegue mostrar.

**Por que Violin Plot?**  
O violin combina o box plot interno com a estimativa de densidade (KDE) nas laterais. A **largura do violino** em cada altura indica a concentração de imóveis naquele valor de aluguel.  
Se um tier tiver dois picos de frequência (por exemplo, imóveis muito baratos e muito caros, sem valores intermediários), o violin revela isso claramente — o box plot não seria capaz de capturar essa bimodalidade.

In [ ]:
# Create a violin plot
fig_violin = px.violin(
    df,
    x='City_Tier',
    y='Rent',
    title='Violin Plot of Rent by City Tier',
    labels={'City_Tier': 'City Tier', 'Rent': 'Rent Amount'},
    category_orders={'City_Tier': sorted(df['City_Tier'].unique())}
)
fig_violin.show()

---
## Seção 4 — Estatísticas Resumidas: Média, Mediana e Desvio Padrão

**Objetivo:** Quantificar numericamente as diferenças entre os tiers e apresentar as métricas mais relevantes de forma comparativa lado a lado.

**Por que comparar Média e Mediana juntas?**  
Se a média for bem maior que a mediana, isso indica que **outliers de alto valor estão puxando a média para cima**, o que distorceria conclusões se apenas a média fosse usada. A mediana representa melhor o "aluguel típico" nesses casos.

**O que o Desvio Padrão alto revela?**  
Um desvio padrão alto em todos os tiers indica que o **nível da cidade sozinho não explica toda a variação do aluguel** — outras variáveis (tamanho do imóvel, bairro, etc.) também contribuem significativamente.

In [ ]:
# Calcula as estatísticas agregadas por tier
stats = df.groupby('City_Tier')['Rent'].agg(
    Media='mean',
    Mediana='median',
    Desvio_Padrao='std',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75)
).reset_index().round(2)

# Três subgráficos lado a lado: Média | Mediana | Desvio Padrão
fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Média do Aluguel', 'Mediana do Aluguel', 'Desvio Padrão'])

colors = px.colors.qualitative.Bold[:3]

for i, (col, titulo) in enumerate(zip(
    ['Media', 'Mediana', 'Desvio_Padrao'],
    ['Média', 'Mediana', 'Desvio Padrão']
), 1):
    fig.add_trace(
        go.Bar(
            x=stats['City_Tier'], y=stats[col],
            marker_color=colors,
            text=stats[col], textposition='outside',
            showlegend=False
        ),
        row=1, col=i
    )

fig.update_layout(
    title='Estatísticas do Aluguel por City_Tier',
    plot_bgcolor='white',
    height=450
)
fig.show()

# Exibe também a tabela completa com Q1 e Q3
print(stats.to_string(index=False))

---
## Seção 5 — Proporção do Rent em Relação à Renda (Rent/Income) por City_Tier

**Objetivo:** Avaliar o **peso financeiro relativo do aluguel** em cada tier, normalizando o valor do aluguel pela renda individual.

**Por que essa seção é a mais analítica do notebook?**  
Comparar o valor absoluto do aluguel entre tiers pode ser enganoso: um aluguel de R$2.000 pode ser leve para quem ganha R$10.000, mas pesado para quem ganha R$3.000.  
A variável `Rent_Income_Ratio` resolve isso ao expressar o aluguel como **percentual da renda**, respondendo à pergunta: *em qual tier as pessoas comprometem mais a renda com moradia?*  
Este indicador é diretamente relevante para análises de saúde financeira pessoal.

In [ ]:
# Criar variável derivada: aluguel como % da renda
df['Rent_Income_Ratio'] = (df['Rent'] / df['Income']) * 100

fig = px.box(
    df, x='City_Tier', y='Rent_Income_Ratio',
    color='City_Tier',
    title='% da Renda Gasta com Aluguel por City_Tier',
    labels={
        'Rent_Income_Ratio': '% Renda → Aluguel',
        'City_Tier': 'Nível da Cidade'
    },
    color_discrete_sequence=px.colors.qualitative.Bold,
    points='outliers'  # Destaca casos onde o aluguel compromete uma fatia anormalmente alta da renda
)
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

print("Média do % de renda gasto com aluguel por City_Tier:")
print(df.groupby('City_Tier')['Rent_Income_Ratio'].mean().round(2))

---
## Seção 6 — Conclusões

**Objetivo:** Consolidar os principais achados da análise exploratória, conectando os resultados de cada seção em uma narrativa coerente.

Com base na análise acima, podemos concluir:

- **Tier_1** apresenta os maiores valores médios e medianos de aluguel, seguido por Tier_2 e Tier_3 — o que é esperado, pois cidades maiores tendem a ter custo de vida mais elevado.
- **Tier_2** concentra a maior parte dos registros (~50%), sendo o grupo mais representativo do dataset. Conclusões sobre Tier_2 são, portanto, as mais robustas estatisticamente.
- O **desvio padrão** é alto em todos os tiers, indicando grande variabilidade nos valores de aluguel mesmo dentro de um mesmo nível de cidade — o tier sozinho não é suficiente para prever o aluguel.
- A **relação Rent/Income** indica quanto da renda é comprometida com aluguel: cidades Tier_1 tendem a ter essa proporção mais alta, revelando maior pressão financeira sobre os moradores.

> **Implicação para o modelo de ML:** A variável `City_Tier` tem poder preditivo sobre o aluguel, mas a alta variância interna de cada tier sugere que outras variáveis (como tamanho do imóvel, renda ou composição familiar) devem ser incluídas no modelo para melhorar a acurácia.